prove/unprove
this notebook is to prove whether result of LLM query make better vector search

In Stanford education video there is a diccussion that using the result of vanila llm query of a question and chain it to vectordb can produce better retrieved data. 
the reason is the distance similirity result of llm query of particular question is closer to chunks in vector db than the question itself even when llm hallucinate.

In [ ]:
question="what is the republic?"
ground_truth="""
The Republic is a Socratic dialogue written by Plato around 375 BC. It is one of the most influential works of philosophy and political theory in history.
"""

In [1]:
from langchain_openai import ChatOpenAI
from pydantic import SecretStrxxxx

LLM_SERVER='http://0.0.0.0:8081/v1'
LLM_API_KEY='sk-no-key-required'
LLM_MODEL='ibm-granite/granite-4.0-micro'

#LLM_SERVER='https://openrouter.ai/api/v1'
#LLM_API_KEY='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' #os.environ["OPENROUTER_API_KEY"]
#LLM_MODEL='x-ai/grok-4.1-fast:free'
#LLM_MODEL='mistralai/mistral-7b-instruct:free'

llm = ChatOpenAI(base_url=LLM_SERVER,
                 temperature=1,
                 api_key=SecretStr(LLM_API_KEY),
                 max_tokens=10000,
                 extra_body={"thinking": {"type": "enabled","budget_tokens": 16000}}
                )

/home/hagusta/conda/envs/simple_rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [5]:

messages = [
    ("user", q),
]
ai_msg = llm.invoke(messages)
print(ai_msg.content)

The term "republic" refers to a type of government where the country is considered a "commonwealth" or "community," and power resides in the people. In a republic, citizens elect representatives who make decisions on their behalf, typically through an elected president or prime minister.

Key characteristics of a republic include:

1. Citizenship: All members of the community have equal rights and responsibilities.
2. Sovereignty: The ultimate authority lies with the people, not the ruler.
3. Limited government: A constitution usually limits the power of the government to protect individual rights and freedoms.
4. Free elections: Leaders are chosen by free and fair elections rather than hereditary succession or absolute monarchies.

Examples of republics around the world include the United States, France, India, Brazil, and Japan. These countries have elected officials who serve in a democratic system where citizens can vote for their representatives.


In [8]:
q_vec=embeddings.embed_query(q)
llm_q_vec=embeddings.embed_query(ai_msg.content)

In [11]:
import numpy as np

def cosine_similarity_numpy(vec1, vec2):
    """
    Calculates the cosine similarity between two numpy vectors.
    """
    # Ensure inputs are numpy arrays
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    # Calculate the dot product (numerator)
    dot_product = np.dot(vec1, vec2)

    # Calculate the L2 norms (magnitudes) of both vectors
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)

    # Handle the case where one or both vectors are zero vectors to avoid division by zero
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0
        
    # Calculate the cosine similarity
    similarity = dot_product / (norm_vec1 * norm_vec2)

    return similarity

# Example Usage
vector_a = [1, 2, 3, 4]
vector_b = [5, 6, 7, 8]

sim = cosine_similarity_numpy(vector_a, vector_b)
print(f"The cosine similarity between the two vectors is: {sim}")

The cosine similarity between the two vectors is: 0.9688639316269662


In [12]:
cosine_similarity_numpy(q_vec,llm_q_vec)

np.float64(0.6516445451357448)

In [14]:
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

In [15]:
collection_name='simple_rag'

qdrant = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=collection_name,
    url="http://localhost:6333",
    content_payload_key="text",
)

retriever = qdrant.as_retriever(search_kwargs={"k" : 2})

In [16]:
q_search=retriever.invoke(q)

In [18]:
llm_q_search=retriever.invoke(ai_msg.content)

In [ ]:
q_search_vec=embeddings.embed_query(q_search)
llm_q_search_vec=embeddings.embed_query(llm_q_search)

In [19]:
q_search

[Document(metadata={'_id': 348, '_collection_name': 'simple_rag'}, page_content='5. For the relation of the Republic to the Statesman and the Laws, and\nthe two other works of Plato which directly treat of politics, see the\nIntroductions to the two latter; a few general points of comparison may\nbe touched upon in this place.\n\nAnd first of the Laws.'),
 Document(metadata={'_id': 8145, '_collection_name': 'simple_rag'}, page_content="Monarchies are superior than Republics I'll argue that a monarchical form of government is much better than a republican form of government on the following issues:- Political stability- Political corruption - Economy- LegitimacyAny questions about this topics please write it con comments.The first round is for acceptanceSecond for main argumentsThird for rebuttals and, if possible, new arguments. Fourth again for rebuttals and new arguments.Fifth for rebuttals and conclusion.")]

In [20]:
llm_q_search

[Document(metadata={'_id': 3824, '_collection_name': 'simple_rag'}, page_content='Democracy is the best form of government. Webster\'s Dictionary defines democracy as:"government by the people; especially: rule of the majority."Whereas a Republic is defined as:"a state in which supreme power is held by the people and their elected representatives, and which has an elected or nominated president rather than a monarch."Since I said in Round 1 that I support a Constitutional Republic, the existence of a Constitution which specifies the the structure of the government, the powers thereof, and the rights of the people. That said, I will argue that Representative Democracy, with a Constitution that specifies the powers of the government and the procedures thereof, is superior to absolute direct democracy. Let me demonstrate the difference- a perfect example of true democracy is the lynch mob- under this system, if someone is accused of a crime, the mob says lynch him so they lynch him. Under

In [21]:
ai_msg.content

'The term "republic" refers to a type of government where the country is considered a "commonwealth" or "community," and power resides in the people. In a republic, citizens elect representatives who make decisions on their behalf, typically through an elected president or prime minister.\n\nKey characteristics of a republic include:\n\n1. Citizenship: All members of the community have equal rights and responsibilities.\n2. Sovereignty: The ultimate authority lies with the people, not the ruler.\n3. Limited government: A constitution usually limits the power of the government to protect individual rights and freedoms.\n4. Free elections: Leaders are chosen by free and fair elections rather than hereditary succession or absolute monarchies.\n\nExamples of republics around the world include the United States, France, India, Brazil, and Japan. These countries have elected officials who serve in a democratic system where citizens can vote for their representatives.'